# BiologicalProcess → ChemicalEntity Relation Pipeline

Builds a unified, deduplicated edge table for the **BiologicalProcess–ChemicalEntity** relation  
by ingesting processed files from multiple KG sources, normalising identifiers  
via PubChem mapping dictionaries, and writing the final triple table to disk.

**Output schema:** `head | relation | tail | head_type | relation_type | tail_type | kg_source | head_id_is | tail_id_is | head_detail_name | tail_detail_name`

---
**Sources included:**
- Monarch
- PrimeKG

**Base paths:**
- KG data: `/storage/Arushi/090526_EvoAge/kg_formation/`
- Mapping databases: `/storage/Arushi/090526_EvoAge/kg_formation/data_collection/databases_for_mapping/`
- Processed KG files: `/storage/Arushi/090526_EvoAge/kg_formation/processed_data/`

In [37]:
#

## 0 · Imports & Base Paths

In [1]:
import pandas as pd
import os

# ── Base directories ──────────────────────────────────────────────────────────
BASE_DIR     = '/storage/Arushi/090526_EvoAge/kg_formation/'
MAPPING_DIR  = BASE_DIR + 'data_collection/databases_for_mapping/'
PROC_DIR     = BASE_DIR + 'processed_data/'

# ── Output path ───────────────────────────────────────────────────────────────
OUT_PATH = BASE_DIR + 'processed_data_relation_wise_merge/generalised/BIOLOGICALPROCESS_CHEMICALENTITY/ALL_BIOLOGICALPROCESS_CHEMICALENTITY.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'kg_source'
]

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


## 1 · Load Chemical Identifier Mapping Dictionaries

Mapping resources loaded:
1. **PubChem combined_df.pkl**: used to map CID to IUPAC names and SMILES.
2. **MESH_GO_ID_NAME.csv**: used to map GO IDs to Biological Process names.

In [35]:
# ── PubChem ───────────────────────────────────────────────────────────────────
Pubchem = pd.read_parquet(MAPPING_DIR + 'pubchem/combined_df.paraquet')
Pubchem_IUPAC_CID_Dict  = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_IUPAC_NAME']))
Pubchem_CID_Smile_Dict  = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_SMILES']))
del Pubchem

# ── MESH GO Dictionary ────────────────────────────────────────────────────────
GO = pd.read_csv(MAPPING_DIR + 'MESH/MESH_GO_ID_NAME.csv')
GO_dict = dict(zip(GO['id'], GO['name']))
del GO

print(f"PubChem IUPAC maps loaded: {len(Pubchem_IUPAC_CID_Dict):,}")
print(f"GO terms loaded: {len(GO_dict):,}")

PubChem IUPAC maps loaded: 119,177,440
GO terms loaded: 47,995


## 2 · Load Source Files

Each source DataFrame is loaded, columns lowercased, and IDs configured.

### 2.1 Monarch

In [39]:
# /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Human/Human_BiologicalProcess_C

In [3]:
Monarch_BP_CE = pd.read_csv(PROC_DIR + 'Monarch/Monarch_final/Human/BiologicalProcess_ChemicalEntity.csv')
Monarch_BP_CE.columns = Monarch_BP_CE.columns.str.lower()

Monarch_BP_CE['head_id_is'] = 'Quick_GO'
Monarch_BP_CE['tail_id_is'] = 'PubChem'
Monarch_BP_CE['relation'] = 'BiologicalProcess_ChemicalEntity'
Monarch_BP_CE['head_type'] = 'BiologicalProcess'
Monarch_BP_CE['tail_type'] = 'ChemicalEntity'

Monarch_BP_CE['kg_source'] = 'Monarch_KG'
Monarch_BP_CE['kg_type'] = 'Generalised'
Monarch_BP_CE['species'] = 'Homo sapiens'

print(f"Monarch BP→CE: {len(Monarch_BP_CE):,} rows")
Monarch_BP_CE
Monarch_BP_CE = Monarch_BP_CE.rename(columns={
    'head_name': 'head_detail_name',
    'tail_name': 'tail_detail_name'
})
Monarch_BP_CE

Monarch BP→CE: 2,662 rows


,head,tail,relation_type,relation_source,kgsource,head_detail_name,head_type,head_id_is,head_taxon,head_taxon_label,...,relation,species_species,head_id_is,relation,head_type,tail_type,tail_id,kg_source,kg_type,species
0,GO:0061370,6013,related_to,infores:go,MonarchKG,testosterone biosynthetic process,BiologicalProcess,Quick_GO,NaN,NaN,...,BiologicalProcess_ChemicalEntity,NaN,Quick_GO,BiologicalProcess_ChemicalEntity,BiologicalProcess,ChemicalEntity,CHEBI:17347,Monarch_KG,Generalised,Homo sapiens
1,GO:0061528,5460542,related_to,infores:go,MonarchKG,aspartate secretion,BiologicalProcess,Quick_GO,NaN,NaN,...,BiologicalProcess_ChemicalEntity,NaN,Quick_GO,BiologicalProcess_ChemicalEntity,BiologicalProcess,ChemicalEntity,CHEBI:35391,Monarch_KG,Generalised,Homo sapiens
2,GO:0061538,6971009,related_to,infores:go,MonarchKG,"histamine secretion, neurotransmission",BiologicalProcess,Quick_GO,NaN,NaN,...,BiologicalProcess_ChemicalEntity,NaN,Quick_GO,BiologicalProcess_ChemicalEntity,BiologicalProcess,ChemicalEntity,CHEBI:57595,Monarch_KG,Generalised,Homo sapiens
3,GO:0061539,21680226,related_to,infores:go,MonarchKG,octopamine secretion,BiologicalProcess,Quick_GO,NaN,NaN,...,BiologicalProcess_ChemicalEntity,NaN,Quick_GO,BiologicalProcess_ChemicalEntity,BiologicalProcess,ChemicalEntity,CHEBI:58025,Monarch_KG,Generalised,Homo sapiens
4,GO:0061545,5249538,related_to,infores:go,MonarchKG,tyramine secretion,BiologicalProcess,Quick_GO,NaN,NaN,...,BiologicalProcess_ChemicalEntity,NaN,Quick_GO,BiologicalProcess_ChemicalEntity,BiologicalProcess,ChemicalEntity,CHEBI:327995,Monarch_KG,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2657,GO:0055075,813,related_to,infores:go,MonarchKG,potassium ion homeostasis,BiologicalProcess,Quick_GO,NaN,NaN,...,BiologicalProcess_ChemicalEntity,NaN,Quick_GO,BiologicalProcess_ChemicalEntity,BiologicalProcess,ChemicalEntity,CHEBI:29103,Monarch_KG,Generalised,Homo sapiens
2658,GO:0055078,923,related_to,infores:go,MonarchKG,sodium ion homeostasis,BiologicalProcess,Quick_GO,NaN,NaN,...,BiologicalProcess_ChemicalEntity,NaN,Quick_GO,BiologicalProcess_ChemicalEntity,BiologicalProcess,ChemicalEntity,CHEBI:29101,Monarch_KG,Generalised,Homo sapiens
2659,GO:0055092,1107,related_to,infores:go,MonarchKG,sterol homeostasis,BiologicalProcess,Quick_GO,NaN,NaN,...,BiologicalProcess_ChemicalEntity,NaN,Quick_GO,BiologicalProcess_ChemicalEntity,BiologicalProcess,ChemicalEntity,CHEBI:15889,Monarch_KG,Generalised,Homo sapiens
2660,GO:0055129,6971047,related_to,infores:go,MonarchKG,L-proline biosynthetic process,BiologicalProcess,Quick_GO,NaN,NaN,...,BiologicalProcess_ChemicalEntity,NaN,Quick_GO,BiologicalProcess_ChemicalEntity,BiologicalProcess,ChemicalEntity,CHEBI:60039,Monarch_KG,Generalised,Homo sapiens


In [111]:
#

### 2.2 PrimeKG

In [95]:
PrimeKG_BP_CE = pd.read_csv(PROC_DIR + 'PrimeKG/PrimeKG_BiologicalProcess_ChemicalEntity_1.csv')
PrimeKG_BP_CE.columns = PrimeKG_BP_CE.columns.str.lower()

PrimeKG_BP_CE['head_id_is'] = 'Quick_GO'
PrimeKG_BP_CE['tail_id_is'] = 'PubChem'
PrimeKG_BP_CE['relation'] = 'BiologicalProcess_ChemicalEntity'
PrimeKG_BP_CE['head_type'] = 'BiologicalProcess'
PrimeKG_BP_CE['tail_type'] = 'ChemicalEntity'

PrimeKG_BP_CE['kg_source'] = 'PrimeKG'

PrimeKG_BP_CE['kg_type'] = 'Generalised'
PrimeKG_BP_CE['species'] = 'Homo sapiens'

# Clean integer tail CID which is saved as float
PrimeKG_BP_CE["tail"] = PrimeKG_BP_CE["tail"].astype(str).str.replace(r'\.0$', '', regex=True)

# We use the MESH dictionary to fill PrimeKG head_detail_name where missing or wrong
if 'head_detail_name' in PrimeKG_BP_CE.columns:
    PrimeKG_BP_CE['head_detail_name'] = PrimeKG_BP_CE['head'].map(GO_dict).fillna(PrimeKG_BP_CE['head_detail_name'])
else:
    PrimeKG_BP_CE['head_detail_name'] = PrimeKG_BP_CE['head'].map(GO_dict)

print(f"PrimeKG BP→CE: {len(PrimeKG_BP_CE):,} rows")
PrimeKG_BP_CE.head(3)

PrimeKG BP→CE: 1,625 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_go_id,tail_detail_name,tail_pubchem_id,tail_smiles_name,tail_smiles_name.1,kg_type,species,head_detail_name
0,GO:0008217,BiologicalProcess_ChemicalEntity,37270,BiologicalProcess,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,regulation of blood pressure,"1,2,3,4,6,7,8-heptachlorodibenzo-p-dioxin","1,2,3,4,6,7,8-heptachlorodibenzodioxin",C1=C2C(=C(C(=C1Cl)Cl)Cl)OC3=C(O2)C(=C(C(=C3Cl)...,C1=C2C(=C(C(=C1Cl)Cl)Cl)OC3=C(O2)C(=C(C(=C3Cl)...,Generalised,Homo sapiens,regulation of blood pressure
1,GO:0008217,BiologicalProcess_ChemicalEntity,38199,BiologicalProcess,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,regulation of blood pressure,"1,2,3,4,6,7,8-heptachlorodibenzofuran","1,2,3,4,6,7,8-heptachlorodibenzofuran",C1=C2C3=C(C(=C(C(=C3Cl)Cl)Cl)Cl)OC2=C(C(=C1Cl)...,C1=C2C3=C(C(=C(C(=C3Cl)Cl)Cl)Cl)OC2=C(C(=C1Cl)...,Generalised,Homo sapiens,regulation of blood pressure
2,GO:0008217,BiologicalProcess_ChemicalEntity,51130,BiologicalProcess,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,regulation of blood pressure,"1,2,3,4,7,8-hexachlorodibenzofuran","1,2,3,4,7,8-hexachlorodibenzofuran",C1=C2C(=CC(=C1Cl)Cl)OC3=C2C(=C(C(=C3Cl)Cl)Cl)Cl,C1=C2C(=CC(=C1Cl)Cl)OC3=C2C(=C(C(=C3Cl)Cl)Cl)Cl,Generalised,Homo sapiens,regulation of blood pressure


## 3 · Consolidate into Unified Schema

All source DataFrames are aligned to `REQUIRED_COLS` (missing columns filled with `pd.NA`),  
then concatenated into `final_df`.

In [97]:
source_dfs = [
    Monarch_BP_CE,
    PrimeKG_BP_CE
   
]

aligned = []

for df in source_dfs:

    # remove duplicate columns
    df = df.loc[:, ~df.columns.duplicated()]

    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None

    aligned.append(df[REQUIRED_COLS])

final_df = pd.concat(aligned, ignore_index=True)
final_df

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,kg_source
0,GO:0061370,BiologicalProcess_ChemicalEntity,6013,BiologicalProcess,related_to,ChemicalEntity,Monarch_KG,Quick_GO,PubChem,testosterone biosynthetic process,testosterone,Generalised,Monarch_KG
1,GO:0061528,BiologicalProcess_ChemicalEntity,5460542,BiologicalProcess,related_to,ChemicalEntity,Monarch_KG,Quick_GO,PubChem,aspartate secretion,aspartate(1-),Generalised,Monarch_KG
2,GO:0061538,BiologicalProcess_ChemicalEntity,6971009,BiologicalProcess,related_to,ChemicalEntity,Monarch_KG,Quick_GO,PubChem,"histamine secretion, neurotransmission",L-histidine zwitterion,Generalised,Monarch_KG
3,GO:0061539,BiologicalProcess_ChemicalEntity,21680226,BiologicalProcess,related_to,ChemicalEntity,Monarch_KG,Quick_GO,PubChem,octopamine secretion,octopaminium,Generalised,Monarch_KG
4,GO:0061545,BiologicalProcess_ChemicalEntity,5249538,BiologicalProcess,related_to,ChemicalEntity,Monarch_KG,Quick_GO,PubChem,tyramine secretion,tyraminium,Generalised,Monarch_KG
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4282,GO:0030222,BiologicalProcess_ChemicalEntity,62705,BiologicalProcess,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,eosinophil differentiation,formaldehyde;urea,Generalised,PrimeKG
4283,GO:0071606,BiologicalProcess_ChemicalEntity,nan,BiologicalProcess,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,chemokine (C-C motif) ligand 4 production,NaN,Generalised,PrimeKG
4284,GO:0007569,BiologicalProcess_ChemicalEntity,nan,BiologicalProcess,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,obsolete cell aging,NaN,Generalised,PrimeKG
4285,GO:0030431,BiologicalProcess_ChemicalEntity,nan,BiologicalProcess,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,sleep,NaN,Generalised,PrimeKG


## 4 · Chemical Name Normalisation (tail)

Rows where `tail_detail_name` is missing are resolved using `Pubchem_IUPAC_CID_Dict` by mapping the Pubchem ID in `tail`.

In [98]:
# Attempt to convert Pubchem ID to float first to resolve potential type mismatch in dict
tail_ids = pd.to_numeric(final_df['tail'], errors='coerce')

mask = final_df['tail_detail_name'].isna()
final_df.loc[mask, 'tail_detail_name'] = tail_ids[mask].map(Pubchem_IUPAC_CID_Dict)

final_df

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,kg_source
0,GO:0061370,BiologicalProcess_ChemicalEntity,6013,BiologicalProcess,related_to,ChemicalEntity,Monarch_KG,Quick_GO,PubChem,testosterone biosynthetic process,testosterone,Generalised,Monarch_KG
1,GO:0061528,BiologicalProcess_ChemicalEntity,5460542,BiologicalProcess,related_to,ChemicalEntity,Monarch_KG,Quick_GO,PubChem,aspartate secretion,aspartate(1-),Generalised,Monarch_KG
2,GO:0061538,BiologicalProcess_ChemicalEntity,6971009,BiologicalProcess,related_to,ChemicalEntity,Monarch_KG,Quick_GO,PubChem,"histamine secretion, neurotransmission",L-histidine zwitterion,Generalised,Monarch_KG
3,GO:0061539,BiologicalProcess_ChemicalEntity,21680226,BiologicalProcess,related_to,ChemicalEntity,Monarch_KG,Quick_GO,PubChem,octopamine secretion,octopaminium,Generalised,Monarch_KG
4,GO:0061545,BiologicalProcess_ChemicalEntity,5249538,BiologicalProcess,related_to,ChemicalEntity,Monarch_KG,Quick_GO,PubChem,tyramine secretion,tyraminium,Generalised,Monarch_KG
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4282,GO:0030222,BiologicalProcess_ChemicalEntity,62705,BiologicalProcess,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,eosinophil differentiation,formaldehyde;urea,Generalised,PrimeKG
4283,GO:0071606,BiologicalProcess_ChemicalEntity,nan,BiologicalProcess,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,chemokine (C-C motif) ligand 4 production,NaN,Generalised,PrimeKG
4284,GO:0007569,BiologicalProcess_ChemicalEntity,nan,BiologicalProcess,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,obsolete cell aging,NaN,Generalised,PrimeKG
4285,GO:0030431,BiologicalProcess_ChemicalEntity,nan,BiologicalProcess,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,sleep,NaN,Generalised,PrimeKG


In [99]:
mask = final_df['head_detail_name'].isna()
final_df.loc[mask, 'head_detail_name'] = tail_ids[mask].map(GO_dict)
final_df

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,kg_source
0,GO:0061370,BiologicalProcess_ChemicalEntity,6013,BiologicalProcess,related_to,ChemicalEntity,Monarch_KG,Quick_GO,PubChem,testosterone biosynthetic process,testosterone,Generalised,Monarch_KG
1,GO:0061528,BiologicalProcess_ChemicalEntity,5460542,BiologicalProcess,related_to,ChemicalEntity,Monarch_KG,Quick_GO,PubChem,aspartate secretion,aspartate(1-),Generalised,Monarch_KG
2,GO:0061538,BiologicalProcess_ChemicalEntity,6971009,BiologicalProcess,related_to,ChemicalEntity,Monarch_KG,Quick_GO,PubChem,"histamine secretion, neurotransmission",L-histidine zwitterion,Generalised,Monarch_KG
3,GO:0061539,BiologicalProcess_ChemicalEntity,21680226,BiologicalProcess,related_to,ChemicalEntity,Monarch_KG,Quick_GO,PubChem,octopamine secretion,octopaminium,Generalised,Monarch_KG
4,GO:0061545,BiologicalProcess_ChemicalEntity,5249538,BiologicalProcess,related_to,ChemicalEntity,Monarch_KG,Quick_GO,PubChem,tyramine secretion,tyraminium,Generalised,Monarch_KG
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4282,GO:0030222,BiologicalProcess_ChemicalEntity,62705,BiologicalProcess,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,eosinophil differentiation,formaldehyde;urea,Generalised,PrimeKG
4283,GO:0071606,BiologicalProcess_ChemicalEntity,nan,BiologicalProcess,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,chemokine (C-C motif) ligand 4 production,NaN,Generalised,PrimeKG
4284,GO:0007569,BiologicalProcess_ChemicalEntity,nan,BiologicalProcess,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,obsolete cell aging,NaN,Generalised,PrimeKG
4285,GO:0030431,BiologicalProcess_ChemicalEntity,nan,BiologicalProcess,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,sleep,NaN,Generalised,PrimeKG


In [100]:
#

In [101]:
# Drop rows where detail names are completely unresolvable
before = len(final_df)
final_df = final_df.dropna(subset=['head_detail_name', 'tail_detail_name']).reset_index(drop=True)
print(f"Dropped {before - len(final_df):,} unresolvable rows → {len(final_df):,} remaining")

Dropped 564 unresolvable rows → 3,723 remaining


## 5 · Sanity Check — Distinct Values

In [102]:
for col in ['relation', 'head_type', 'tail_type', 'relation_type', 'kg_source', 'head_id_is', 'tail_id_is', 'kg_type', 'kg_source']:
    print(f"{col:20s}: {set(final_df[col])}")

relation            : {'BiologicalProcess_ChemicalEntity'}
head_type           : {'BiologicalProcess'}
tail_type           : {'ChemicalEntity'}
relation_type       : {'interacts with', 'related_to'}
kg_source           : {'kg_source'}
head_id_is          : {'Quick_GO'}
tail_id_is          : {'PubChem'}
kg_type             : {'Generalised'}
kg_source           : {'kg_source'}


In [103]:
#

## 6 · NaN Audit (pre-dedup)

In [104]:
true_nan   = final_df.isna().sum()
string_nan = final_df.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
})

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


In [105]:
final_df['species'] = 'Homo sapiens'
final_df

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,kg_source,species
0,GO:0061370,BiologicalProcess_ChemicalEntity,6013,BiologicalProcess,related_to,ChemicalEntity,Monarch_KG,Quick_GO,PubChem,testosterone biosynthetic process,testosterone,Generalised,Monarch_KG,Homo sapiens
1,GO:0061528,BiologicalProcess_ChemicalEntity,5460542,BiologicalProcess,related_to,ChemicalEntity,Monarch_KG,Quick_GO,PubChem,aspartate secretion,aspartate(1-),Generalised,Monarch_KG,Homo sapiens
2,GO:0061538,BiologicalProcess_ChemicalEntity,6971009,BiologicalProcess,related_to,ChemicalEntity,Monarch_KG,Quick_GO,PubChem,"histamine secretion, neurotransmission",L-histidine zwitterion,Generalised,Monarch_KG,Homo sapiens
3,GO:0061539,BiologicalProcess_ChemicalEntity,21680226,BiologicalProcess,related_to,ChemicalEntity,Monarch_KG,Quick_GO,PubChem,octopamine secretion,octopaminium,Generalised,Monarch_KG,Homo sapiens
4,GO:0061545,BiologicalProcess_ChemicalEntity,5249538,BiologicalProcess,related_to,ChemicalEntity,Monarch_KG,Quick_GO,PubChem,tyramine secretion,tyraminium,Generalised,Monarch_KG,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3718,GO:0048806,BiologicalProcess_ChemicalEntity,5564,BiologicalProcess,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,genitalia development,"5-chloro-2-(2,4-dichlorophenoxy)phenol",Generalised,PrimeKG,Homo sapiens
3719,GO:0030217,BiologicalProcess_ChemicalEntity,62705,BiologicalProcess,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,T cell differentiation,formaldehyde;urea,Generalised,PrimeKG,Homo sapiens
3720,GO:0002323,BiologicalProcess_ChemicalEntity,62705,BiologicalProcess,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,natural killer cell activation involved in imm...,formaldehyde;urea,Generalised,PrimeKG,Homo sapiens
3721,GO:0030222,BiologicalProcess_ChemicalEntity,62705,BiologicalProcess,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,eosinophil differentiation,formaldehyde;urea,Generalised,PrimeKG,Homo sapiens


## 7 · Deduplication

Group on `(head, relation, tail)`. For `kg_source`, merge all unique sources with `::` delimiter.  
All other metadata columns take the first non-null value.

In [106]:
# remove duplicate columns (keep first)
final_df = final_df.loc[:, ~final_df.columns.duplicated()]
final_df

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,GO:0061370,BiologicalProcess_ChemicalEntity,6013,BiologicalProcess,related_to,ChemicalEntity,Monarch_KG,Quick_GO,PubChem,testosterone biosynthetic process,testosterone,Generalised,Homo sapiens
1,GO:0061528,BiologicalProcess_ChemicalEntity,5460542,BiologicalProcess,related_to,ChemicalEntity,Monarch_KG,Quick_GO,PubChem,aspartate secretion,aspartate(1-),Generalised,Homo sapiens
2,GO:0061538,BiologicalProcess_ChemicalEntity,6971009,BiologicalProcess,related_to,ChemicalEntity,Monarch_KG,Quick_GO,PubChem,"histamine secretion, neurotransmission",L-histidine zwitterion,Generalised,Homo sapiens
3,GO:0061539,BiologicalProcess_ChemicalEntity,21680226,BiologicalProcess,related_to,ChemicalEntity,Monarch_KG,Quick_GO,PubChem,octopamine secretion,octopaminium,Generalised,Homo sapiens
4,GO:0061545,BiologicalProcess_ChemicalEntity,5249538,BiologicalProcess,related_to,ChemicalEntity,Monarch_KG,Quick_GO,PubChem,tyramine secretion,tyraminium,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3718,GO:0048806,BiologicalProcess_ChemicalEntity,5564,BiologicalProcess,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,genitalia development,"5-chloro-2-(2,4-dichlorophenoxy)phenol",Generalised,Homo sapiens
3719,GO:0030217,BiologicalProcess_ChemicalEntity,62705,BiologicalProcess,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,T cell differentiation,formaldehyde;urea,Generalised,Homo sapiens
3720,GO:0002323,BiologicalProcess_ChemicalEntity,62705,BiologicalProcess,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,natural killer cell activation involved in imm...,formaldehyde;urea,Generalised,Homo sapiens
3721,GO:0030222,BiologicalProcess_ChemicalEntity,62705,BiologicalProcess,interacts with,ChemicalEntity,PrimeKG,Quick_GO,PubChem,eosinophil differentiation,formaldehyde;urea,Generalised,Homo sapiens


In [107]:
def merge_sources(x):
    return '::'.join(sorted(set(x.dropna().astype(str))))

group_cols = ['head', 'relation', 'tail']

final_df_dedup = (
    final_df
    .groupby(group_cols, as_index=False)
    .agg({
        'head_type':        'first',
        'relation_type':    'first',
        'tail_type':        'first',
        'kg_source':        merge_sources,
        'head_id_is':       'first',
        'tail_id_is':       'first',
        'head_detail_name': 'first',
        'tail_detail_name': 'first',
        'kg_type':          merge_sources,
        'species':          'first'
    })
)

print(f"Before dedup: {len(final_df):,}")
print(f"After dedup : {len(final_df_dedup):,}")

Before dedup: 3,723
After dedup : 3,723


## 8 · Post-dedup NaN Audit & Source Distribution

In [108]:
true_nan   = final_df_dedup.isna().sum()
string_nan = final_df_dedup.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

display(pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
}))

print("\nkg_source values present:", set(final_df_dedup['kg_source']))

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0



kg_source values present: {'PrimeKG', 'Monarch_KG'}


## 9 · Save Output

In [113]:
# OUT_PATH

In [110]:
import os
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 3,723 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/BIOLOGICALPROCESS_CHEMICALENTITY/ALL_BIOLOGICALPROCESS_CHEMICALENTITY.csv


In [114]:
!pwd

/storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised
